# Detect Font Sizes in PDF Using PyMuPDF and Draw Boxes with OpenCV

This notebook uses **PyMuPDF** to extract font sizes and bounding boxes of text spans from a PDF, and **OpenCV** to draw rectangles around those spans with font size annotations.

In [ ]:
# Install required packages
!pip install pymupdf opencv-python-headless numpy

In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np


In [ ]:
def detect_font_and_draw_boxes(pdf_path, output_image_path):
    # Open the PDF file
    doc = fitz.open(pdf_path)
    
    # Extract the first page
    page = doc
    
    # Get the page pixmap to convert to image
    pix = page.get_pixmap()
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:  # Convert RGBA to BGR
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    
    # Extract font size and bounding boxes
    blocks = page.get_text('dict')['blocks']
    for block in blocks:
        if block['type'] == 0:  # text block
            for line in block['lines']:
                for span in line['spans']:
                    font_size = span['size']
                    bbox = span['bbox']  # (x0, y0, x1, y1)
                    x0, y0, x1, y1 = map(int, bbox)
                    # Draw rectangle
                    cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
                    # Put font size text
                    cv2.putText(img, str(int(font_size)), (x0, y0 - 5), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    # Save output image
    cv2.imwrite(output_image_path, img)


## Usage Example

Replace `'sample.pdf'` with the path to your PDF file. The output image with annotated boxes will be saved as `'output.png'`.

In [ ]:
pdf_path = 'sample.pdf'
output_image_path = 'output.png'
detect_font_and_draw_boxes(pdf_path, output_image_path)
print(f'Output image saved to {output_image_path}')